<a href="https://colab.research.google.com/github/sumangal01/dr-screening/blob/main/AI_EYE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("No CUDA GPU assigned")

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/dr_project/checkpoints', exist_ok=True)
!nvidia-smi
!pip -q install timm albumentations kaggle
files.upload()  # upload kaggle.json here

In [ ]:
!kaggle datasets download -d mariaherrerot/aptos2019 -p /content/data --unzip
!ls /content/data

In [ ]:
import pandas as pd, os
d = '/content/data'
for f in ['train_1.csv', 'valid.csv', 'test.csv']:
    df = pd.read_csv(f'{d}/{f}')
    print(f, df.shape, list(df.columns))
    print(df.head(3), '\n')
    c = [x for x in df.columns if 'diag' in x.lower() or 'level' in x.lower()]
    if c: print(df[c[0]].value_counts().sort_index(), '\n')
for folder in ['train_images', 'val_images', 'test_images']:
    print(folder, len(os.listdir(f'{d}/{folder}')))
print(os.listdir(f'{d}/train_images')[:3])

In [ ]:
import pandas as pd, os
from PIL import Image
d = '/content/data'
tr = pd.read_csv(f'{d}/train_1.csv'); va = pd.read_csv(f'{d}/valid.csv'); te = pd.read_csv(f'{d}/test.csv')

for name in ['train_images', 'val_images', 'test_images']:
    inner = f'{d}/{name}/{name}'
    print(name, os.path.isdir(inner), len(os.listdir(inner)) if os.path.isdir(inner) else '-')
print(os.listdir(f'{d}/train_images/train_images')[:3])

print('overlap train/valid:', len(set(tr.id_code) & set(va.id_code)))
print('overlap train/test :', len(set(tr.id_code) & set(te.id_code)))
print('overlap valid/test :', len(set(va.id_code) & set(te.id_code)))

f = os.listdir(f'{d}/train_images/train_images')[0]
im = Image.open(f'{d}/train_images/train_images/{f}')
print(f, im.size, im.mode)

In [ ]:
import cv2, numpy as np, pandas as pd, os
from concurrent.futures import ThreadPoolExecutor

D, OUT, S = '/content/data', '/content/cache', 384
splits = {'train': ('train_1.csv', 'train_images'),
          'val':   ('valid.csv',   'val_images'),
          'test':  ('test.csv',    'test_images')}

def prep(args):
    src, dst = args
    img = cv2.imread(src)
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ys, xs = np.where(g > 10)
    if len(ys) > 0:
        img = img[ys.min():ys.max()+1, xs.min():xs.max()+1]
    img = cv2.resize(img, (S, S), interpolation=cv2.INTER_AREA)
    cv2.imwrite(dst, img)

for name, (csv, folder) in splits.items():
    os.makedirs(f'{OUT}/{name}', exist_ok=True)
    df = pd.read_csv(f'{D}/{csv}')
    jobs = [(f'{D}/{folder}/{folder}/{i}.png', f'{OUT}/{name}/{i}.png') for i in df.id_code]
    with ThreadPoolExecutor(4) as ex: list(ex.map(prep, jobs))
    print(name, len(os.listdir(f'{OUT}/{name}')))

In [ ]:
import torch, timm, cv2, numpy as np, pandas as pd, os
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import cohen_kappa_score

D, C = '/content/data', '/content/cache'
CK = '/content/drive/MyDrive/dr_project/checkpoints'
dev = 'cuda'
EPOCHS, BS, LR = 15, 32, 3e-4
torch.manual_seed(42); np.random.seed(42)

tr = pd.read_csv(f'{D}/train_1.csv'); va = pd.read_csv(f'{D}/valid.csv')
norm = [A.Normalize(), ToTensorV2()]
aug_tr = A.Compose([A.HorizontalFlip(), A.VerticalFlip(),
                    A.Rotate(limit=180, border_mode=cv2.BORDER_CONSTANT, p=0.7),
                    A.RandomBrightnessContrast(0.2, 0.2, p=0.5)] + norm)
aug_va = A.Compose(norm)

class DS(Dataset):
    def __init__(s, df, folder, tf): s.df, s.f, s.tf = df.reset_index(drop=True), folder, tf
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]
        im = cv2.cvtColor(cv2.imread(f'{C}/{s.f}/{r.id_code}.png'), cv2.COLOR_BGR2RGB)
        return s.tf(image=im)['image'], int(r.diagnosis)

dl_tr = DataLoader(DS(tr, 'train', aug_tr), BS, shuffle=True, num_workers=2, drop_last=True)
dl_va = DataLoader(DS(va, 'val', aug_va), BS, num_workers=2)

model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(dev)
cnt = tr.diagnosis.value_counts().sort_index().values
w = 1 / np.sqrt(cnt); w = torch.tensor(w / w.sum() * 5, dtype=torch.float32).to(dev)
crit = torch.nn.CrossEntropyLoss(weight=w)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
scaler = torch.cuda.amp.GradScaler()
start, best = 0, -1

if os.path.exists(f'{CK}/last.pt'):
    ck = torch.load(f'{CK}/last.pt', map_location=dev)
    model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched']); start, best = ck['epoch'] + 1, ck['best']
    print('resumed from epoch', start)

for ep in range(start, EPOCHS):
    model.train(); tl = 0
    for x, y in dl_tr:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad()
        with torch.autocast('cuda'): loss = crit(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tl += loss.item()
    sched.step()
    model.eval(); P, Y = [], []
    with torch.no_grad(), torch.autocast('cuda'):
        for x, y in dl_va:
            P += model(x.to(dev)).argmax(1).cpu().tolist(); Y += y.tolist()
    q = cohen_kappa_score(Y, P, weights='quadratic')
    acc = np.mean(np.array(P) == np.array(Y))
    print(f'ep {ep+1}/{EPOCHS} loss {tl/len(dl_tr):.3f} val QWK {q:.3f} acc {acc:.3f}')
    if q > best:
        best = q; torch.save(model.state_dict(), f'{CK}/best.pt')
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                'sched': sched.state_dict(), 'epoch': ep, 'best': best}, f'{CK}/last.pt')
print('best val QWK', best)

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, cohen_kappa_score
import torch.nn.functional as F

te = pd.read_csv(f'{D}/test.csv')
dl_te = DataLoader(DS(te, 'test', aug_va), BS, num_workers=2)
model.load_state_dict(torch.load(f'{CK}/best.pt', map_location=dev)); model.eval()

probs, Y = [], []
with torch.no_grad(), torch.autocast('cuda'):
    for x, y in dl_te:
        probs.append(F.softmax(model(x.to(dev)).float(), 1).cpu().numpy()); Y += y.tolist()
probs, Y = np.concatenate(probs), np.array(Y)
P = probs.argmax(1)

print('Test QWK :', round(cohen_kappa_score(Y, P, weights='quadratic'), 3))
print('Test acc :', round((P == Y).mean(), 3))
cm = confusion_matrix(Y, P, labels=range(5)); print('\nConfusion matrix (rows=true, cols=pred)\n', cm)
print('\nPer-grade recall:')
for g in range(5): print(f'  grade {g}: {cm[g, g]}/{cm[g].sum()} = {cm[g, g]/cm[g].sum():.2f}')

yt, yp = Y >= 2, P >= 2
sens = (yt & yp).sum() / yt.sum(); spec = (~yt & ~yp).sum() / (~yt).sum()
auc = roc_auc_score(yt, probs[:, 2:].sum(1))
print(f'\nReferable DR (grade>=2): sensitivity {sens:.3f}, specificity {spec:.3f}, AUC {auc:.3f}')

np.save(f'{CK}/test_probs.npy', probs)

In [ ]:
note = """DR baseline v1 (EfficientNet-B0, 384px, APTOS 2019)
Val QWK 0.907 (best epoch)
Test QWK 0.893, acc 0.847
Referable DR: sens 0.942, spec 0.965, AUC 0.990
Grade recall: 0:0.98 1:0.67 2:0.83 3:0.47 4:0.42
Setup: 15 epochs, batch 32, lr 3e-4, AdamW, weighted CE loss, seed 42
Dataset: kaggle mariaherrerot/aptos2019 (pre-split 2930/366/366)
"""
open('/content/drive/MyDrive/dr_project/results_v1.txt', 'w').write(note)

In [ ]:
import matplotlib.pyplot as plt, torch.nn.functional as F
model.eval()
acts, grads = {}, {}

def fwd(m, i, o):
    acts['a'] = o
    o.register_hook(lambda g: grads.__setitem__('g', g))
model.conv_head.register_forward_hook(fwd)

def gradcam(i):
    r = te.iloc[i]
    im = cv2.cvtColor(cv2.imread(f'/content/cache/test/{r.id_code}.png'), cv2.COLOR_BGR2RGB)
    x = aug_va(image=im)['image'].unsqueeze(0).to(dev)
    model.zero_grad()
    out = model(x); c = out.argmax(1).item()
    out[0, c].backward()
    w = grads['g'].mean((2, 3), keepdim=True)
    m = F.relu((w * acts['a']).sum(1))[0].detach().cpu().numpy()
    m = (m - m.min()) / (m.max() - m.min() + 1e-8)
    m = cv2.resize(m, (im.shape[1], im.shape[0]))
    heat = cv2.cvtColor(cv2.applyColorMap((m * 255).astype('uint8'), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(im, 0.55, heat, 0.45, 0), int(r.diagnosis), c

true = te.diagnosis.values
pred = probs.argmax(1)
picks = [int(np.where((true == g) & (pred == g))[0][0]) for g in [0, 2, 3, 4]]
picks += list(np.where(true != pred)[0][:4])

fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, i in zip(ax.ravel(), picks):
    img, t, p = gradcam(i)
    a.imshow(img); a.axis('off')
    a.set_title(f'true grade {t} | AI said {p}', color='green' if t == p else 'red')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/dr_project/gradcam_examples.png', dpi=120)
plt.show()

In [ ]:
!kaggle datasets download -d USERNAME/DATASET-NAME -p /content/idrid --unzip
!find /content/idrid -maxdepth 3 | head -40

In [ ]:
!kaggle datasets list -s IDRiD

In [ ]:
!kaggle datasets download -d mariaherrerot/idrid-dataset -p /content/idrid --unzip
!find /content/idrid -maxdepth 3 | head -40

In [ ]:
import pandas as pd, glob
for f in glob.glob('/content/idrid/**/*.csv', recursive=True):
    df = pd.read_csv(f)
    print(f, df.shape, list(df.columns))
    print(df.head(3), '\n')

In [ ]:
import os, cv2, numpy as np, pandas as pd, torch, timm, torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, roc_auc_score, cohen_kappa_score

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

C = '/content/cache'; CK = '/content/drive/MyDrive/dr_project/checkpoints'; dev = 'cuda'
aug_va = A.Compose([A.Normalize(), ToTensorV2()])

class DS(Dataset):
    def __init__(s, df, folder, tf): s.df, s.f, s.tf = df.reset_index(drop=True), folder, tf
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]
        im = cv2.cvtColor(cv2.imread(f'{C}/{s.f}/{r.id_code}.png'), cv2.COLOR_BGR2RGB)
        return s.tf(image=im)['image'], int(r.diagnosis)

lab = pd.read_csv('/content/idrid/idrid_labels.csv').iloc[:, :2]
lab.columns = ['id_code', 'diagnosis']
lab = lab.dropna(); lab['diagnosis'] = lab.diagnosis.astype(int)
print('cached IDRiD images:', len(os.listdir(f'{C}/idrid')))

model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=5).to(dev)
model.load_state_dict(torch.load(f'{CK}/best.pt', map_location=dev)); model.eval()

dl = DataLoader(DS(lab, 'idrid', aug_va), 32, num_workers=2)
probs, Y = [], []
with torch.no_grad(), torch.autocast('cuda'):
    for x, y in dl:
        probs.append(F.softmax(model(x.to(dev)).float(), 1).cpu().numpy()); Y += y.tolist()
probs, Y = np.concatenate(probs), np.array(Y); P = probs.argmax(1)

print('IDRiD external QWK:', round(cohen_kappa_score(Y, P, weights='quadratic'), 3))
print('IDRiD external acc:', round((P == Y).mean(), 3))
cm = confusion_matrix(Y, P, labels=range(5)); print('\nConfusion matrix (rows=true, cols=pred)\n', cm)
for g in range(5):
    if cm[g].sum(): print(f'  grade {g}: {cm[g,g]}/{cm[g].sum()} = {cm[g,g]/cm[g].sum():.2f}')
yt, yp = Y >= 2, P >= 2
print(f'\nReferable DR: sensitivity {(yt&yp).sum()/yt.sum():.3f}, specificity {(~yt&~yp).sum()/(~yt).sum():.3f}, AUC {roc_auc_score(yt, probs[:,2:].sum(1)):.3f}')
np.save(f'{CK}/idrid_probs.npy', probs)

In [ ]:
import os, shutil
print('cache folders:', {d: len(os.listdir(f'/content/cache/{d}')) for d in os.listdir('/content/cache')} if os.path.exists('/content/cache') else 'NO CACHE')

!mkdir -p /content/data
!kaggle datasets download -d mariaherrerot/aptos2019 -f train_1.csv -p /content/data --unzip
!kaggle datasets download -d mariaherrerot/aptos2019 -f valid.csv -p /content/data --unzip
!kaggle datasets download -d mariaherrerot/aptos2019 -f test.csv -p /content/data --unzip
print(os.listdir('/content/data'))

B = '/content/drive/MyDrive/dr_project/data_backup'
os.makedirs(B, exist_ok=True)
shutil.make_archive(f'{B}/cache', 'zip', '/content', 'cache')
for f in ['train_1.csv', 'valid.csv', 'test.csv']:
    shutil.copy(f'/content/data/{f}', B)
shutil.copy('/content/idrid/idrid_labels.csv', B)
open('/content/drive/MyDrive/dr_project/results_v1.txt', 'w').write("""DR baseline v1 (EfficientNet-B0, 384px, trained on APTOS 2019)
APTOS test : QWK 0.893, acc 0.847, referable sens 0.942, spec 0.965, AUC 0.990
IDRiD ext. : QWK 0.783, acc 0.530, referable sens 0.806, spec 0.980, AUC 0.980
IDRiD grade recall: 0:0.71 1:0.77 2:0.62 3:0.20 4:0.31
Setup: 15 epochs, batch 32, lr 3e-4, AdamW, weighted CE, seed 42
""")
for f in os.listdir(B): print(f, round(os.path.getsize(f'{B}/{f}') / 1e6, 1), 'MB')

In [ ]:
import json, time, os, numpy as np
RES = '/content/drive/MyDrive/dr_project/results'
os.makedirs(RES, exist_ok=True)

def save_result(name, metrics, probs=None, labels=None):
    stamp = time.strftime('%Y%m%d_%H%M')
    with open(f'{RES}/{name}_{stamp}.json', 'w') as f:
        json.dump(metrics, f, indent=2, default=float)
    if probs is not None: np.save(f'{RES}/{name}_{stamp}_probs.npy', probs)
    if labels is not None: np.save(f'{RES}/{name}_{stamp}_labels.npy', labels)
    print('saved:', name, stamp)

In [ ]:
save_result('idrid_v1', {'qwk': 0.783, 'acc': 0.530, 'ref_sens': 0.806, 'ref_spec': 0.980}, probs, Y)

In [ ]:
import numpy as np, pandas as pd, json, time, os
from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

CK = '/content/drive/MyDrive/dr_project/checkpoints'
RES = '/content/drive/MyDrive/dr_project/results'; os.makedirs(RES, exist_ok=True)
def save_result(name, metrics, probs=None, labels=None):
    stamp = time.strftime('%Y%m%d_%H%M')
    json.dump(metrics, open(f'{RES}/{name}_{stamp}.json', 'w'), indent=2, default=float)
    if probs is not None: np.save(f'{RES}/{name}_{stamp}_probs.npy', probs)
    if labels is not None: np.save(f'{RES}/{name}_{stamp}_labels.npy', labels)
    print('saved:', name, stamp)

probs = np.load(f'{CK}/idrid_probs.npy')
lab = pd.read_csv('/content/idrid/idrid_labels.csv').iloc[:, :2]; lab.columns = ['id_code', 'diagnosis']
Y = lab.dropna().diagnosis.astype(int).values
assert len(Y) == len(probs), 'size mismatch'

ia, ib = train_test_split(np.arange(len(Y)), test_size=0.5, stratify=Y, random_state=42)
logit = np.log(probs + 1e-9)

# 1) temperature scaling, fitted on half A only
nll = lambda T, i: -np.log(softmax(logit[i] / T, 1)[np.arange(len(i)), Y[i]] + 1e-9).mean()
T = minimize_scalar(lambda t: nll(t, ia), bounds=(0.3, 5), method='bounded').x
pc = softmax(logit / T, 1)

def ece(p, y, n=10):
    conf, pred = p.max(1), p.argmax(1); e = 0
    for lo in np.linspace(0, 1, n, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1 / n)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e

# 2) referable-DR threshold, chosen on half A to reach about 95% sensitivity
rf = lambda p: p[:, 2:].sum(1)
yt = Y >= 2
cand = np.linspace(0.02, 0.9, 200)
ok = [t for t in cand if (rf(pc[ia]) >= t)[yt[ia]].mean() >= 0.95]
thr = max(ok) if ok else 0.5

def report(tag, p, idx, t):
    pred = rf(p[idx]) >= t; y = yt[idx]
    s, sp = (pred & y).sum() / y.sum(), (~pred & ~y).sum() / (~y).sum()
    print(f'{tag}: sensitivity {s:.3f}, specificity {sp:.3f}'); return s, sp

print(f'temperature T = {T:.2f} | referable threshold = {thr:.2f}')
print(f'ECE (confidence error) on half B: before {ece(probs[ib], Y[ib]):.3f} -> after {ece(pc[ib], Y[ib]):.3f}\n')
b = report('Half B BEFORE (threshold 0.50)', probs, ib, 0.5)
a = report(f'Half B AFTER  (threshold {thr:.2f})', pc, ib, thr)

save_result('idrid_calibration_v1',
    {'T': T, 'threshold': thr, 'halfB_before_sens_spec': b, 'halfB_after_sens_spec': a,
     'ece_before': ece(probs[ib], Y[ib]), 'ece_after': ece(pc[ib], Y[ib])})

In [ ]:
import os, glob
B = '/content/drive/MyDrive/dr_project/data_backup'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

print('now in /content/cache:', os.listdir('/content/cache') if os.path.exists('/content/cache') else 'NOTHING')
print('Drive backup folder  :', os.listdir(B) if os.path.exists(B) else 'NOT FOUND')

if os.path.exists(f'{B}/cache.zip'):
    !unzip -q -o {B}/cache.zip -d /content
    print({d: len(os.listdir(f'/content/cache/{d}')) for d in os.listdir('/content/cache')})

In [ ]:
import os
B = '/content/drive/MyDrive/dr_project/data_backup'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

print('now in /content/cache:', os.listdir('/content/cache') if os.path.exists('/content/cache') else 'NOTHING')
print('Drive backup folder  :', os.listdir(B) if os.path.exists(B) else 'NOT FOUND')

if os.path.exists(f'{B}/cache.zip'):
    !unzip -q -o {B}/cache.zip -d /content
    print({d: len(os.listdir(f'/content/cache/{d}')) for d in os.listdir('/content/cache')})

In [ ]:
import os, cv2, numpy as np, pandas as pd, shutil
from concurrent.futures import ThreadPoolExecutor
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')

B = '/content/drive/MyDrive/dr_project/data_backup'; os.makedirs(B, exist_ok=True)
D, R, OUT = '/content/data', '/content/idrid', '/content/cache'

if os.path.exists(f'{B}/cache.zip'):
    print('Restoring from Drive backup...')
    !unzip -q -o {B}/cache.zip -d /content
    print({d: len(os.listdir(f'{OUT}/{d}')) for d in os.listdir(OUT)})
else:
    print('No backup found. Rebuilding (about 8 minutes)...')
    if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        from google.colab import files; _ = files.upload()
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip -q install kaggle
    !kaggle datasets download -d mariaherrerot/aptos2019 -p {D} --unzip
    !kaggle datasets download -d mariaherrerot/idrid-dataset -p {R} --unzip

    S = 384
    def prep(a):
        src, dst = a
        img = cv2.imread(src); g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        ys, xs = np.where(g > 10)
        if len(ys): img = img[ys.min():ys.max()+1, xs.min():xs.max()+1]
        cv2.imwrite(dst, cv2.resize(img, (S, S), interpolation=cv2.INTER_AREA))

    jobs = []
    for name, csv, folder in [('train', 'train_1.csv', 'train_images'), ('val', 'valid.csv', 'val_images'), ('test', 'test.csv', 'test_images')]:
        os.makedirs(f'{OUT}/{name}', exist_ok=True)
        jobs += [(f'{D}/{folder}/{folder}/{i}.png', f'{OUT}/{name}/{i}.png') for i in pd.read_csv(f'{D}/{csv}').id_code]
    lab = pd.read_csv(f'{R}/idrid_labels.csv').iloc[:, :2]; lab.columns = ['id_code', 'diagnosis']; lab = lab.dropna()
    os.makedirs(f'{OUT}/idrid', exist_ok=True)
    jobs += [(f'{R}/Imagenes/Imagenes/{i}.jpg', f'{OUT}/idrid/{i}.png') for i in lab.id_code]
    with ThreadPoolExecutor(4) as ex: list(ex.map(prep, jobs))

    shutil.make_archive(f'{B}/cache', 'zip', '/content', 'cache')
    for f in ['train_1.csv', 'valid.csv', 'test.csv']: shutil.copy(f'{D}/{f}', B)
    shutil.copy(f'{R}/idrid_labels.csv', B)
    print({d: len(os.listdir(f'{OUT}/{d}')) for d in os.listdir(OUT)})

print('Drive backup now contains:', os.listdir(B))

In [ ]:
import os, cv2, numpy as np, pandas as pd, shutil
from concurrent.futures import ThreadPoolExecutor

B = '/content/drive/MyDrive/dr_project/data_backup'; D = '/content/data'; OUT = '/content/cache'
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    from google.colab import files; _ = files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
!kaggle datasets download -d mariaherrerot/aptos2019 -p {D} --unzip

def prep(a):
    src, dst = a
    img = cv2.imread(src); g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ys, xs = np.where(g > 10)
    if len(ys): img = img[ys.min():ys.max()+1, xs.min():xs.max()+1]
    cv2.imwrite(dst, cv2.resize(img, (384, 384), interpolation=cv2.INTER_AREA))

jobs = []
for name, csv, folder in [('train', 'train_1.csv', 'train_images'), ('val', 'valid.csv', 'val_images'), ('test', 'test.csv', 'test_images')]:
    os.makedirs(f'{OUT}/{name}', exist_ok=True)
    jobs += [(f'{D}/{folder}/{folder}/{i}.png', f'{OUT}/{name}/{i}.png') for i in pd.read_csv(f'{D}/{csv}').id_code]
with ThreadPoolExecutor(4) as ex: list(ex.map(prep, jobs))

counts = {d: len(os.listdir(f'{OUT}/{d}')) for d in os.listdir(OUT)}
print(counts)
assert counts == {'train': 2930, 'val': 366, 'test': 366, 'idrid': 455}, 'counts wrong, NOT backing up'

shutil.make_archive(f'{B}/cache', 'zip', '/content', 'cache')
print('backup saved:', round(os.path.getsize(f'{B}/cache.zip') / 1e6), 'MB')

In [ ]:
import os
P = '/content/drive/MyDrive/dr_project'
for root, _, files in os.walk(P):
    for f in sorted(files):
        print(os.path.join(root, f).replace(P, '').lstrip('/'), '|', round(os.path.getsize(os.path.join(root, f)) / 1e6, 1), 'MB')

In [ ]:
%%writefile /content/drive/MyDrive/dr_project/dr_utils.py
import os, json, time, cv2, numpy as np, pandas as pd, torch, timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

C = '/content/cache'
CK = '/content/drive/MyDrive/dr_project/checkpoints'
RES = '/content/drive/MyDrive/dr_project/results'
dev = 'cuda'
os.makedirs(RES, exist_ok=True)
aug_va = A.Compose([A.Normalize(), ToTensorV2()])

class DS(Dataset):
    def __init__(s, df, folder, tf): s.df, s.f, s.tf = df.reset_index(drop=True), folder, tf
    def __len__(s): return len(s.df)
    def __getitem__(s, i):
        r = s.df.iloc[i]
        im = cv2.cvtColor(cv2.imread(f'{C}/{s.f}/{r.id_code}.png'), cv2.COLOR_BGR2RGB)
        return s.tf(image=im)['image'], int(r.diagnosis)

def load_model():
    m = timm.create_model('efficientnet_b0', pretrained=False, num_classes=5).to(dev)
    m.load_state_dict(torch.load(f'{CK}/best.pt', map_location=dev)); m.eval()
    return m

def save_result(name, metrics, probs=None, labels=None):
    stamp = time.strftime('%Y%m%d_%H%M')
    json.dump(metrics, open(f'{RES}/{name}_{stamp}.json', 'w'), indent=2, default=float)
    if probs is not None: np.save(f'{RES}/{name}_{stamp}_probs.npy', probs)
    if labels is not None: np.save(f'{RES}/{name}_{stamp}_labels.npy', labels)
    print('saved:', name, stamp)

In [ ]:
import cv2, numpy as np, pandas as pd, os, matplotlib.pyplot as plt
C = '/content/cache'; RES = '/content/drive/MyDrive/dr_project/results'
os.makedirs(RES, exist_ok=True)
rows = []
for folder in ['test', 'idrid']:
    for f in os.listdir(f'{C}/{folder}'):
        im = cv2.imread(f'{C}/{folder}/{f}'); g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        m = g > 15
        rows.append({'set': folder, 'file': f,
                     'sharpness': cv2.Laplacian(g, cv2.CV_64F).var(),
                     'brightness': g[m].mean() if m.any() else 0,
                     'retina_area': m.mean()})
q = pd.DataFrame(rows)
print(q.groupby('set')[['sharpness', 'brightness', 'retina_area']].describe().T.round(1).to_string())

q['flag'] = (q.sharpness < q.sharpness.quantile(0.05)) | (q.brightness < 40) | (q.brightness > 200) | (q.retina_area < 0.6)
print('\nflagged:', q.flag.sum(), 'of', len(q))
q.to_csv(f'{RES}/quality_scores_v1.csv', index=False)

worst = q.sort_values('sharpness').head(8)
fig, ax = plt.subplots(2, 4, figsize=(16, 8))
for a, r in zip(ax.ravel(), worst.itertuples()):
    a.imshow(cv2.cvtColor(cv2.imread(f'{C}/{r.set}/{r.file}'), cv2.COLOR_BGR2RGB)); a.axis('off')
    a.set_title(f'{r.set} sharp={r.sharpness:.0f} bright={r.brightness:.0f}', fontsize=9)
plt.tight_layout(); plt.savefig(f'{RES}/quality_worst8_v1.png', dpi=110); plt.show()
print('saved to Drive: quality_scores_v1.csv, quality_worst8_v1.png')

In [ ]:
import pandas as pd, numpy as np
CK = '/content/drive/MyDrive/dr_project/checkpoints'; RES = '/content/drive/MyDrive/dr_project/results'
q = pd.read_csv(f'{RES}/quality_scores_v1.csv'); q['id_code'] = q.file.str.replace('.png', '', regex=False)

te = pd.read_csv('/content/data/test.csv'); te['pred'] = np.load(f'{CK}/test_probs.npy').argmax(1); te['set'] = 'test'
lab = pd.read_csv('/content/idrid/idrid_labels.csv').iloc[:, :2]; lab.columns = ['id_code', 'diagnosis']
lab = lab.dropna(); lab['diagnosis'] = lab.diagnosis.astype(int)
lab['pred'] = np.load(f'{CK}/idrid_probs.npy').argmax(1); lab['set'] = 'idrid'

d = pd.concat([te, lab]).merge(q[['set', 'id_code', 'sharpness', 'brightness', 'flag']], on=['set', 'id_code'])
d['correct'] = d.pred == d.diagnosis
d['missed_referable'] = (d.diagnosis >= 2) & (d.pred < 2)
print(d.groupby(['set', 'flag']).agg(n=('correct', 'size'), accuracy=('correct', 'mean'),
                                      missed_referable=('missed_referable', 'sum')).round(3))
print('\nsharpness vs brightness correlation:', round(d.sharpness.corr(d.brightness), 2))
d.to_csv(f'{RES}/quality_vs_accuracy_v1.csv', index=False)
print('saved: quality_vs_accuracy_v1.csv')

In [ ]:
import cv2, numpy as np, pandas as pd, os
C = '/content/cache'; RES = '/content/drive/MyDrive/dr_project/results'
d = pd.read_csv(f'{RES}/quality_vs_accuracy_v1.csv')

def nsharp(r):
    g = cv2.cvtColor(cv2.imread(f'{C}/{r.set}/{r.id_code}.png'), cv2.COLOR_BGR2GRAY).astype(np.float64)
    m = g > 15
    g = g / (g[m].mean() + 1e-6)
    return cv2.Laplacian(g[96:288, 96:288], cv2.CV_64F).var()

d['nsharp'] = [nsharp(r) for r in d.itertuples()]
print('correlation with brightness:', round(d.nsharp.corr(d.brightness), 2))
d['flag2'] = d.nsharp < d.nsharp.quantile(0.10)
print(d.groupby(['set', 'flag2']).agg(n=('correct', 'size'), accuracy=('correct', 'mean'),
                                       missed_referable=('missed_referable', 'sum')).round(3))
d.to_csv(f'{RES}/quality_vs_accuracy_v2.csv', index=False)
print('saved: quality_vs_accuracy_v2.csv')

In [ ]:
%run /content/drive/MyDrive/dr_project/dr_utils.py
import torch.nn.functional as F, matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

T, THR = 3.58, 0.28
NAMES = ['0 No DR', '1 Mild', '2 Moderate', '3 Severe', '4 Proliferative']
model = load_model(); acts, grads = {}, {}
def fwd(m, i, o):
    acts['a'] = o; o.register_hook(lambda g: grads.__setitem__('g', g))
model.conv_head.register_forward_hook(fwd)

def analyse(path):
    im = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    x = aug_va(image=im)['image'].unsqueeze(0).to(dev)
    model.zero_grad(); out = model(x)
    p = F.softmax(out.detach() / T, 1)[0].cpu().numpy()
    out[0, int(out.argmax(1))].backward()
    w = grads['g'].mean((2, 3), keepdim=True)
    m = F.relu((w * acts['a']).sum(1))[0].detach().cpu().numpy()
    m = cv2.resize((m - m.min()) / (m.max() - m.min() + 1e-8), im.shape[:2][::-1])
    heat = cv2.cvtColor(cv2.applyColorMap((m * 255).astype('uint8'), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    return im, cv2.addWeighted(im, 0.55, heat, 0.45, 0), p

def make_report(path, name, true=None):
    im, cam, p = analyse(path)
    ref = p[2:].sum(); refer = ref >= THR
    fig = plt.figure(figsize=(8.27, 11.69))
    fig.text(0.5, 0.96, 'Experimental DR Screening Report', ha='center', fontsize=18, weight='bold')
    fig.text(0.5, 0.935, f'Image: {name}  |  RESEARCH PROTOTYPE, NOT A MEDICAL DEVICE', ha='center', fontsize=9, color='red')
    for k, (img, t) in enumerate([(im, 'Retinal photo'), (cam, 'Attention map (Grad-CAM)')]):
        a = fig.add_axes([0.06 + k * 0.47, 0.60, 0.42, 0.30]); a.imshow(img); a.axis('off'); a.set_title(t, fontsize=10)
    a = fig.add_axes([0.12, 0.36, 0.80, 0.18])
    a.barh(NAMES[::-1], p[::-1] * 100, color='#4a7ebb'); a.set_xlim(0, 100); a.set_xlabel('Calibrated confidence (%)')
    a.set_title('Confidence for each severity grade', fontsize=10)
    verdict = 'REFERABLE: professional eye examination recommended' if refer else 'Not flagged as referable (still follow routine screening advice)'
    txt = (f'Most likely grade: {NAMES[int(p.argmax())]}  ({p.max() * 100:.0f}% confidence)\n'
           f'Referable DR score (grade 2 or higher): {ref:.2f}  (flag threshold {THR})\n'
           f'Result: {verdict}\n' + (f'Reference label (research only): grade {true}\n' if true is not None else '') +
           '\nLimits: trained on APTOS 2019 only; accuracy dropped on a second dataset (IDRiD).\n'
           'Severe grades (3 and 4) are often under-estimated. The attention map shows where the model\n'
           'looked, not proof of disease. No image-quality check is applied. Detects signs of diabetic\n'
           'retinopathy only; it does not diagnose diabetes or other eye diseases.')
    fig.text(0.07, 0.08, txt, fontsize=9.5, va='bottom', linespacing=1.5)
    return fig

os.makedirs('/content/drive/MyDrive/dr_project/reports', exist_ok=True)
te = pd.read_csv('/content/data/test.csv')
for g in [0, 2, 4]:
    r = te[te.diagnosis == g].iloc[0]
    fig = make_report(f'{C}/test/{r.id_code}.png', r.id_code, true=g)
    with PdfPages(f'/content/drive/MyDrive/dr_project/reports/report_{r.id_code}.pdf') as pdf: pdf.savefig(fig)
    plt.show()
print('saved 3 reports to Drive: dr_project/reports')